# MiniMind-Lab 训练监控面板\n\n实时可视化训练指标：loss、学习率。训练脚本写入 `metrics.csv`，本 notebook 读取并绘图。

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import clear_output, display
import time

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

CSV_PATH = "../metrics.csv"
REFRESH_SEC = 5

print('✅ 已加载，运行下一个 Cell 开始监控')

In [ ]:
def plot_metrics(csv_path):
    if not Path(csv_path).exists():
        print(f'⏳ 等待 CSV 文件: {csv_path}')
        return
    
    df = pd.read_csv(csv_path)
    if len(df) < 1:
        print('⏳ CSV 为空，等待数据...')
        return
    
    clear_output(wait=True)
    
    # 找出要绘制的数值列
    numeric_cols = [c for c in df.columns if c not in ('timestamp', 'step', 'epoch')]
    
    n = len(numeric_cols)
    if n == 0:
        print('⚠️ 没有可绘制的数值列')
        return
    
    rows = (n + 1) // 2
    fig, axes = plt.subplots(rows, 2, figsize=(14, 3.5 * rows))
    axes = axes.flatten() if n > 1 else [axes]
    
    for i, col in enumerate(numeric_cols):
        ax = axes[i]
        ax.plot(df['step'], df[col], linewidth=0.8, alpha=0.9)
        ax.set_xlabel('Step')
        ax.set_ylabel(col)
        ax.set_title(f'{col} over Steps')
        ax.grid(True, alpha=0.3)
        
        # 添加 rolling mean
        if len(df) > 10:
            window = min(20, len(df) // 3)
            df[f'{col}_smooth'] = df[col].rolling(window, min_periods=1).mean()
            ax.plot(df['step'], df[f'{col}_smooth'], linewidth=1.5, alpha=1.0, color='red')
    
    # 隐藏多余的 subplot
    for j in range(n, len(axes)):
        axes[j].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # 打印最新指标
    latest = df.iloc[-1]
    print(f'📊 Step {int(latest["step"])} | ', end='')
    for col in numeric_cols:
        print(f'{col}: {latest[col]:.4f} | ', end='')
    print(f'({len(df)} 条记录)')
    print(f'⏱️  下次刷新: {REFRESH_SEC}s 后')

print('✅ 绘图函数已定义')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

print('🟢 开始实时监控... (按 ■ 停止按钮中断)')
print(f'📁 读取: {CSV_PATH}')
print()

try:
    while True:
        plot_metrics(CSV_PATH)
        time.sleep(REFRESH_SEC)
except KeyboardInterrupt:
    print('\n⏹️  监控已停止')